# Experiment 5 — Integrated UTM Scenario

A small deterministic scenario combines four UAVs, fixed obstacles, a static geofence, a dynamic geofence, wind, sensor noise, strategic deconfliction, tracking, and tactical deconfliction.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from musas.airspace import (
    Airspace, Box3D, Cylinder3D, Geofence, Obstacle, WindZone,
)
from musas.algorithms import (
    DetourPathPlanner, FirstOrderDynamics, FlightPlanAwareTracking,
    GaussianSensor, HybridGeometricManeuver,
    MinimumSeparationConflictPrediction,
    MinimumSeparationTacticalConflictPrediction,
    RecursiveTimeShiftDeconfliction, WaypointGuidance,
)
from musas.domain import FlightPlan, Waypoint
from musas.simulation import (
    Simulation, SimulationDataset, VisualizationConfig,
    animate_dataset, calculate_metrics, plot_dataset,
)

OUTPUT_DIRECTORY = Path("outputs") / "output_exp5"
DATASET_DIRECTORY = OUTPUT_DIRECTORY / "datasets"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
DATASET_DIRECTORY.mkdir(exist_ok=True)

In [ ]:
MINIMUM_SEPARATION_DISTANCE = 30.0
COLLISION_DISTANCE = 5.0
SIMULATION_END = 100.0

airspace = Airspace()
airspace.add_obstacle(Obstacle(
    "central-building", Box3D(-30, 30, -25, 25, 0, 125),
    "Central building",
))
airspace.add_obstacle(Obstacle(
    "east-tower", Cylinder3D(95, 55, 18, 0, 145),
    "East tower",
))
airspace.add_geofence(Geofence(
    "hospital", Cylinder3D(-85, 75, 25, 70, 155),
    name="Hospital zone",
))

flight_plans = {
    "UAV A": FlightPlan([
        Waypoint(-180, 0, 100, 4), Waypoint(180, 0, 100, 64),
    ], priority=2),
    "UAV B": FlightPlan([
        Waypoint(0, -170, 145, 7), Waypoint(0, 170, 145, 67),
    ], priority=1),
    "UAV C": FlightPlan([
        Waypoint(-180, 75, 110, 10), Waypoint(180, 75, 110, 70),
    ], priority=1),
    "UAV D": FlightPlan([
        Waypoint(70, -170, 100, 12), Waypoint(70, 170, 100, 72),
    ], priority=0),
}

In [ ]:
dataset_path = DATASET_DIRECTORY / "integrated_scenario.sqlite"
strategic_predictor = MinimumSeparationConflictPrediction(
    MINIMUM_SEPARATION_DISTANCE, 0.5,
)
tactical_predictor = MinimumSeparationTacticalConflictPrediction(
    MINIMUM_SEPARATION_DISTANCE, 0.5,
)
simulation = Simulation(
    airspace=airspace,
    strategic_conflict_prediction_algorithm=strategic_predictor,
    strategic_deconfliction_algorithm=RecursiveTimeShiftDeconfliction(
        conflict_prediction_algorithm=strategic_predictor,
        path_planner=DetourPathPlanner(
            conflict_prediction_algorithm=strategic_predictor,
            lateral_clearance=12.0, vertical_clearance=12.0,
            maximum_maneuvers=6,
        ),
        time_shift_step=4.0, maximum_time_shifts=30,
    ),
    tracking_algorithm=FlightPlanAwareTracking(),
    tactical_conflict_prediction_algorithm=tactical_predictor,
    tactical_deconfliction_algorithm=HybridGeometricManeuver(
        conflict_prediction_algorithm=tactical_predictor,
        minimum_separation_distance=MINIMUM_SEPARATION_DISTANCE,
        vertical_offset=50.0, airspace_clearance=12.0,
        maneuver_duration=5.0, maximum_speed=14.0, cooldown=12.0,
    ),
    report_interval=0.5, history_window=6,
    prediction_horizon=18, prediction_step=0.5,
    tactical_assessment_interval=0.25,
    flight_update_interval=0.1, trajectory_record_interval=0.2,
    collision_distance=COLLISION_DISTANCE, network_delay=0.1,
    dataset_path=dataset_path, overwrite_dataset=True,
    dataset_metadata={"experiment": "integrated_scenario"},
)
simulation.schedule_geofence(Geofence(
    "emergency-zone", Cylinder3D(65, -70, 30, 70, 155),
    name="Temporary emergency zone",
), 28.0, 65.0)
simulation.schedule_wind_zone(WindZone(
    "west-crosswind", Box3D(-200, -20, -160, 160, 70, 170),
    (0.0, 3.0, 0.0), name="West crosswind",
), 8.0, 75.0)

for index, (name, plan) in enumerate(flight_plans.items()):
    simulation.add_uav(
        name=name, flight_plan=plan,
        dynamics_algorithm=FirstOrderDynamics(5.0, 1.0),
        guidance_algorithm=WaypointGuidance(min_speed=0.0, max_speed=14.0),
        sensor_model=GaussianSensor(0.8, 0.15, seed=2026 + index),
        submission_time=0.25 * index, waypoint_acceptance_radius=2.0,
    )
simulation.run(until=SIMULATION_END)
simulation.close()

In [ ]:
with SimulationDataset(dataset_path) as dataset:
    metrics = calculate_metrics(
        dataset, MINIMUM_SEPARATION_DISTANCE, COLLISION_DISTANCE,
    )
{key: metrics[key] for key in (
    "flights", "detected_strategic_conflicts",
    "detected_tactical_conflicts", "modified_flights",
    "tactical_interventions", "loss_of_separation_events",
    "collision_events", "geofence_violations",
    "obstacle_violations",
)}

In [ ]:
config = VisualizationConfig(
    uav_names=None, view="xy", show_real_trajectories=True,
    show_current_positions=True, show_flight_plans=True,
    show_tracks=True, show_sensor_reports=False,
    show_geofences=True, show_wind_zones=True, show_obstacles=True,
    show_legend=False, figure_size=(9.0, 7.0), frame_step=0.5,
    title="Integrated UTM scenario",
)
for timestamp in (20.0, 35.0, 55.0, 80.0):
    image_path = OUTPUT_DIRECTORY / f"t_{timestamp:04.1f}.png"
    figure = plot_dataset(
        dataset_path, config, timestamp, image_path, dpi=180,
    )
    display(figure)

animation_path = OUTPUT_DIRECTORY / "integrated_scenario.gif"
animate_dataset(
    dataset_path, config, animation_path, fps=6.0, dpi=90,
)
display(Image(filename=animation_path))